# 英西翻译推理 Notebook
加载已保存的模型权重、配置和词汇表，还原 Transformer 模型，并进行翻译演示。

In [1]:
import paddle
import numpy as np
import json
import pickle
import string
import re

c:\Users\zetad\anaconda3\envs\es\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [2]:
# Load configuration from JSON file
with open('transformer_config.json', 'r') as f:
    config = json.load(f)

embed_dim      = config['embed_dim']
latent_dim     = config['latent_dim']
num_heads      = config['num_heads']
sequence_length = config['sequence_length']
vocab_size     = config['vocab_size']
START_TOKEN    = config['START_TOKEN']
END_TOKEN      = config['END_TOKEN']
PAD_TOKEN      = config['PAD_TOKEN']
UNK_TOKEN      = config['UNK_TOKEN']

print('Config loaded:', config)

Config loaded: {'embed_dim': 256, 'latent_dim': 2048, 'num_heads': 8, 'sequence_length': 20, 'vocab_size': 30000, 'START_TOKEN': '[start]', 'END_TOKEN': '[end]', 'PAD_TOKEN': '<pad>', 'UNK_TOKEN': '<unk>'}


In [3]:
# Load vocabulary data from pickle file
with open('transformer_vocab.pkl', 'rb') as f:
    vocab_data = pickle.load(f)

eng2id_dict = vocab_data['eng2id_dict']
spa2id_dict = vocab_data['spa2id_dict']
id2eng_dict = vocab_data['id2eng_dict']
id2spa_dict = vocab_data['id2spa_dict']

print(f'English vocabulary size: {len(eng2id_dict)}')
print(f'Spanish vocabulary size: {len(spa2id_dict)}')

English vocabulary size: 11678
Spanish vocabulary size: 22107


In [4]:
# Define the Transformer model components

class TransformerEncoder(paddle.nn.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.attention = paddle.nn.MultiHeadAttention(num_heads=num_heads, embed_dim=embed_dim, dropout=0.1)
        self.dense_proj = paddle.nn.Sequential(
            paddle.nn.Linear(embed_dim, dense_dim),
            paddle.nn.ReLU(),
            paddle.nn.Linear(dense_dim, embed_dim)
        )
        self.layernorm_1 = paddle.nn.LayerNorm(embed_dim)
        self.layernorm_2 = paddle.nn.LayerNorm(embed_dim)

    def forward(self, inputs, mask=None):
        padding_mask = None
        if mask is not None:
            padding_mask = paddle.cast(mask[:, np.newaxis, np.newaxis, :], dtype='int32')
        attention_output = self.attention(query=inputs, value=inputs, key=inputs, attn_mask=padding_mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)


class PositionalEmbedding(paddle.nn.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embeddings = paddle.nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        self.position_embeddings = paddle.nn.Embedding(num_embeddings=sequence_length, embedding_dim=embed_dim)
        self.sequence_length = sequence_length

    def forward(self, inputs):
        length = inputs.shape[-1]
        positions = paddle.arange(start=0, end=length, step=1)
        return self.token_embeddings(inputs) + self.position_embeddings(positions)

    def compute_mask(self, inputs, mask=None):
        return paddle.not_equal(inputs, 0)


class TransformerDecoder(paddle.nn.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads):
        super().__init__()
        self.attention_1 = paddle.nn.MultiHeadAttention(num_heads=num_heads, embed_dim=embed_dim)
        self.attention_2 = paddle.nn.MultiHeadAttention(num_heads=num_heads, embed_dim=embed_dim)
        self.dense_proj = paddle.nn.Sequential(
            paddle.nn.Linear(embed_dim, latent_dim),
            paddle.nn.ReLU(),
            paddle.nn.Linear(latent_dim, embed_dim)
        )
        self.layernorm_1 = paddle.nn.LayerNorm(embed_dim)
        self.layernorm_2 = paddle.nn.LayerNorm(embed_dim)
        self.layernorm_3 = paddle.nn.LayerNorm(embed_dim)

    def forward(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        padding_mask = None
        if mask is not None:
            padding_mask = paddle.cast(mask[:, np.newaxis, :], dtype='int32')
            padding_mask = paddle.minimum(padding_mask, causal_mask)
        attention_output_1 = self.attention_1(query=inputs, value=inputs, key=inputs, attn_mask=causal_mask)
        out_1 = self.layernorm_1(inputs + attention_output_1)
        attention_output_2 = self.attention_2(
            query=out_1, value=encoder_outputs, key=encoder_outputs, attn_mask=padding_mask
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)
        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        seq_len = inputs.shape[1]
        i = paddle.arange(seq_len)[:, np.newaxis]
        j = paddle.arange(seq_len)
        mask = paddle.cast(i >= j, dtype='int32')
        return paddle.reshape(mask, (1, 1, seq_len, seq_len))


class Transformer(paddle.nn.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, sequence_length, vocab_size):
        super().__init__()
        self.ps1 = PositionalEmbedding(sequence_length, vocab_size, embed_dim)
        self.encoder = TransformerEncoder(embed_dim, latent_dim, num_heads)
        self.ps2 = PositionalEmbedding(sequence_length, vocab_size, embed_dim)
        self.decoder = TransformerDecoder(embed_dim, latent_dim, num_heads)
        self.drop = paddle.nn.Dropout(p=0.2)
        self.lastLinear = paddle.nn.Linear(embed_dim, vocab_size)

    def forward(self, encoder_inputs, decoder_inputs):
        encoder_emb = self.ps1(encoder_inputs)
        encoder_outputs = self.encoder(encoder_emb)
        decoder_emb = self.ps2(decoder_inputs)
        decoder_outputs = self.decoder(decoder_emb, encoder_outputs)
        out = self.drop(decoder_outputs)
        return self.lastLinear(out)


print('Model structure defined successfully')

Model structure defined successfully


In [5]:
# Load model weights from .pdparams file
trans = Transformer(embed_dim, latent_dim, num_heads, sequence_length, vocab_size)
state_dict = paddle.load('transformer_model_final.pdparams')
trans.set_state_dict(state_dict)
trans.eval()
print('✅ Model weights loaded successfully')

✅ Model weights loaded successfully


In [6]:
# Define helper functions

import re as _re
_proper_tag_pat = _re.compile(r'(<proper\d+>)(\S+)')

def pre_process(datas, save_punctuation=True):
    dataset = []
    strip_chars = string.punctuation + '¿¡'
    strip_chars = strip_chars.replace('[', '').replace(']', '')
    for text in datas:
        tags_found = []
        def _replace(m):
            tag = m.group(1)
            n = tag[7:-1]
            tags_found.append(n)
            return f'PROPTOKEN{n}'
        processed = _proper_tag_pat.sub(_replace, text)
        lowercase = processed.lower()
        out = ''
        if save_punctuation:
            for low in lowercase:
                if low in strip_chars:
                    out += ' ' + low if low not in ('¿', '¡') else low + ' '
                else:
                    out += low
        else:
            for low in lowercase:
                if low not in strip_chars:
                    out += low
        for n in tags_found:
            out = out.replace(f'proptoken{n}', f'<proper{n}>')
        dataset.append(out)
    return dataset


def build_tensor(data, dicta, maxlen):
    tensor = []
    for text in data:
        subtensor = []
        for word in text.split():
            idx = dicta.get(word, 1)
            subtensor.append(idx)
        if len(subtensor) < maxlen:
            subtensor += [0] * (maxlen - len(subtensor))
        else:
            subtensor = subtensor[:maxlen]
        tensor.append(subtensor)
    return np.array(tensor)


print('Helper functions defined successfully')

Helper functions defined successfully


In [7]:
import string

def _translate_sentence(english_text):
    """Translate a single sentence, return translated string"""
    import re as _re
    _strip = string.punctuation + '¿¡'
    _strip = _strip.replace('[', '').replace(']', '')
    def _tokenize(text):
        out = ''
        for c in text:
            if c not in _strip: out += c
        return out.split()
    # 1. Identify proper nouns
    raw_words = _tokenize(english_text)
    tag_to_word = {}
    counter = 1
    seen = {}
    marked_text = english_text
    for w in raw_words:
        if w[0].isupper() and eng2id_dict.get(w.lower(), 1) == 1:
            if w not in seen:
                seen[w] = counter
                tag_to_word[f'<proper{counter}>'] = w
                counter += 1
            n = seen[w]
            # Replace with pure placeholder (no original word, avoid re-matching)
            marked_text = _re.sub(r'\b' + _re.escape(w) + r'\b', f'XPLACEHOLDERX{n}X', marked_text)
    # Restore placeholder to <properN>word form (keep trailing punctuation, avoid being consumed by _proper_tag_pat)
    for w, n in seen.items():
        marked_text = _re.sub(
            r'XPLACEHOLDERX' + str(n) + r'X([^\w\s]*)',
            lambda m, _w=w, _n=n: f'<proper{_n}>{_w}' + (' ' + m.group(1) if m.group(1) else ''),
            marked_text
        )

    # 2. Preprocess: <properN>word converted to <properN> token, keep punctuation
    preprocessed_text = pre_process([marked_text], save_punctuation=True)[0]
    # 3. Vectorize and run inference
    eng_tensor_arr = build_tensor([preprocessed_text], eng2id_dict, sequence_length)[0]
    eng_tensor = paddle.to_tensor(eng_tensor_arr, dtype='int64')
    encoder_input = paddle.unsqueeze(eng_tensor, axis=0)
    decoded_tokens = [2]
    decoded_words = []

    with paddle.no_grad():
        for i in range(sequence_length - 1):
            decoder_input = paddle.to_tensor([decoded_tokens], dtype='int64')
            if decoder_input.shape[1] < sequence_length:
                pad_len = sequence_length - decoder_input.shape[1]
                decoder_input = paddle.concat([
                    decoder_input,
                    paddle.zeros([1, pad_len], dtype='int64')
                ], axis=1)
            logits = trans(encoder_input, decoder_input)
            current_pos = min(i, sequence_length - 1)
            logits_current = logits[0, current_pos, :]
            probs = paddle.nn.functional.softmax(logits_current)
            sampled_token_index = paddle.argmax(probs).item()
            if sampled_token_index == 3:
                break
            decoded_tokens.append(sampled_token_index)
            sampled_token = id2spa_dict.get(sampled_token_index, UNK_TOKEN)
            if sampled_token not in [PAD_TOKEN, START_TOKEN, END_TOKEN, UNK_TOKEN]:
                # When <properN> token appears in output, look up original word from table
                if sampled_token in tag_to_word:
                    decoded_words.append(tag_to_word[sampled_token])
                else:
                    decoded_words.append(sampled_token)

    decoded_sentence = ' '.join(decoded_words)
    # Spanish paired punctuation: add ¿ if ? present without ¿, add ¡ if ! present without ¡, and vice versa
    if '?' in decoded_sentence and '¿' not in decoded_sentence:
        decoded_sentence = '¿' + decoded_sentence
    elif '¿' in decoded_sentence and '?' not in decoded_sentence:
        decoded_sentence = decoded_sentence + '?'
    if '!' in decoded_sentence and '¡' not in decoded_sentence:
        decoded_sentence = '¡' + decoded_sentence
    elif '¡' in decoded_sentence and '!' not in decoded_sentence:
        decoded_sentence = decoded_sentence + '!'
    import re as _re2
    decoded_sentence = _re2.sub(r'(^[¿¡]*\s*|(?<=[.?!])\s+[¿¡]*\s*)(\S)', lambda m: m.group(1) + m.group(2).upper(), decoded_sentence)
    return decoded_sentence

def translate_custom_english(english_text):
    """Translate sentence by sentence, supports multi-sentence input"""
    import re as _re3
    # Split sentences on .!?, keep delimiters
    parts = _re3.split(r'(?<=[.!?])\s+', english_text.strip())
    translated_parts = []
    for part in parts:
        if part.strip():
            translated_parts.append(_translate_sentence(part.strip()))
    result = ' '.join(translated_parts)
    # Capitalize first letter of sentence and after punctuation
    import re as _re4
    result = _re4.sub(r'(^[¿¡]*\s*|(?<=[.?!])\s+[¿¡]*\s*)(\S)', lambda m: m.group(1) + m.group(2).upper(), result)
    return result
print("\n" + "="*60)
print("🎯 English-Spanish Translation Demo")
print("="*60)

# Example translations
print("\n📝 Example 1: Translate sentence with common words")
test_sentence1 = "I like eating apples"
print(f"Input: {test_sentence1}")
result1 = translate_custom_english(test_sentence1)
print(f"Output: {result1}\n")

print("📝 Example 2: Translate sentence with proper nouns")
test_sentence2 = "I love playing Genshin"
print(f"Input: {test_sentence2}")
result2 = translate_custom_english(test_sentence2)
print(f"Output: {result2}\n")

# Interactive translation
print("\n" + "="*60)
print("🌐 Interactive translation mode")
print("="*60)
print("Please enter an English sentence to translate (type 'exit' to quit)")
print("-"*60)

while True:
    user_input = input("\nEnglish input: ").strip()
    
    if user_input.lower() == 'exit':
        print("✅ Exiting translation program")
        break
    
    if not user_input:
        print("⚠️  Please enter a non-empty English sentence")
        continue
    
    try:
        print(f"Processing...")
        translation = translate_custom_english(user_input)
        print(f"   Translation: {translation}")
    except Exception as e:
        print(f"❌ Translation error: {str(e)}")


🎯 English-Spanish Translation Demo

📝 Example 1: Translate sentence with common words
Input: I like eating apples
Output: Me gusta comer manzanas .

📝 Example 2: Translate sentence with proper nouns
Input: I love playing Genshin
Output: Me encanta jugar Genshin


🌐 Interactive translation mode
Please enter an English sentence to translate (type 'exit' to quit)
------------------------------------------------------------
Processing...
   Translation: ¡ Amo Genshin !
✅ Exiting translation program
